# Delta Load and Analytical Publishing

This notebook publishes quality-approved E-SMAT datasets for analytical consumption.

## Responsibilities

1. Load reusable functions;
2. Configure Silver, quality, and Gold tables;
3. Confirm that the Silver quality gate passed;
4. Read the validated Silver datasets;
5. Create the safe/unsafe classification fact table;
6. Prepare stable Gold datasets for reporting;
7. Validate analytical schemas and row-level expectations;
8. Persist Gold Delta tables;
9. Produce a publishing summary.

## Outputs

- `inspection_header`
- `inspection_responses`
- `inspection_requirements`
- `inspection_classification`

The published tables are intended for Power BI and other governed analytical consumers.

> Production catalog and schema names are represented by placeholders in this portfolio version.


## 1. Load shared functions

In [ ]:
%run ./01_shared_functions


## 2. Configure sources and analytical targets

In [ ]:
SILVER_CATALOG = "<catalog>"
SILVER_SCHEMA = "<silver-schema>"
QUALITY_CATALOG = "<catalog>"
QUALITY_SCHEMA = "<quality-schema>"
GOLD_CATALOG = "<catalog>"
GOLD_SCHEMA = "<gold-schema>"

SILVER_TABLES = {
    "inspection_header": (
        f"{SILVER_CATALOG}.{SILVER_SCHEMA}.inspection_header_silver"
    ),
    "inspection_responses": (
        f"{SILVER_CATALOG}.{SILVER_SCHEMA}.inspection_responses_silver"
    ),
    "inspection_requirements": (
        f"{SILVER_CATALOG}.{SILVER_SCHEMA}.inspection_requirements_silver"
    ),
}

QUALITY_TABLES = {
    "summary": (
        f"{QUALITY_CATALOG}.{QUALITY_SCHEMA}.data_quality_summary"
    ),
    "failures": (
        f"{QUALITY_CATALOG}.{QUALITY_SCHEMA}.data_quality_failures"
    ),
}

GOLD_TABLES = {
    "inspection_header": (
        f"{GOLD_CATALOG}.{GOLD_SCHEMA}.inspection_header"
    ),
    "inspection_responses": (
        f"{GOLD_CATALOG}.{GOLD_SCHEMA}.inspection_responses"
    ),
    "inspection_requirements": (
        f"{GOLD_CATALOG}.{GOLD_SCHEMA}.inspection_requirements"
    ),
    "inspection_classification": (
        f"{GOLD_CATALOG}.{GOLD_SCHEMA}.inspection_classification"
    ),
}

WRITE_MODE = "overwrite"
OVERWRITE_SCHEMA = "true"


## 3. Validate publishing configuration

In [ ]:
def validate_publishing_config(
    silver_tables,
    quality_tables,
    gold_tables,
):
    required_silver = {
        "inspection_header",
        "inspection_responses",
        "inspection_requirements",
    }
    required_quality = {"summary", "failures"}
    required_gold = {
        "inspection_header",
        "inspection_responses",
        "inspection_requirements",
        "inspection_classification",
    }

    missing_configuration = {
        "silver": sorted(required_silver - set(silver_tables)),
        "quality": sorted(required_quality - set(quality_tables)),
        "gold": sorted(required_gold - set(gold_tables)),
    }
    missing_configuration = {
        layer: values
        for layer, values in missing_configuration.items()
        if values
    }

    if missing_configuration:
        raise ValueError(
            f"Missing Delta-load configuration: {missing_configuration}"
        )

    all_table_names = [
        *silver_tables.values(),
        *quality_tables.values(),
        *gold_tables.values(),
    ]

    for table_name in all_table_names:
        if not table_name or table_name.count(".") != 2:
            raise ValueError(
                f"Invalid fully qualified table name: {table_name}"
            )

    if len(gold_tables.values()) != len(set(gold_tables.values())):
        raise ValueError("Gold target tables must be unique.")

    print("Delta-load configuration validated successfully.")


validate_publishing_config(
    SILVER_TABLES,
    QUALITY_TABLES,
    GOLD_TABLES,
)


## 4. Confirm the Silver quality gate

The analytical load is permitted only when the quality-summary table exists and contains no failed checks with severity `error`.


In [ ]:
quality_summary_table = QUALITY_TABLES["summary"]

if not spark.catalog.tableExists(quality_summary_table):
    raise RuntimeError(
        "Data-quality summary table was not found. "
        "Run 04_data_quality.ipynb before publishing Gold tables."
    )

quality_summary_df = spark.table(quality_summary_table)

require_columns(
    dataframe=quality_summary_df,
    required_columns={
        "check_id",
        "severity",
        "status",
        "failure_count",
        "processing_timestamp",
    },
    dataset_name="data_quality_summary",
)

latest_quality_timestamp = quality_summary_df.agg(
    F.max("processing_timestamp").alias("latest_processing_timestamp")
).first()["latest_processing_timestamp"]

if latest_quality_timestamp is None:
    raise RuntimeError(
        "Data-quality summary does not contain an execution timestamp."
    )

latest_quality_df = quality_summary_df.filter(
    F.col("processing_timestamp") == F.lit(latest_quality_timestamp)
)

blocking_quality_df = latest_quality_df.filter(
    (F.col("severity") == "error")
    & (F.col("status") == "failed")
)

blocking_quality_count = blocking_quality_df.count()

if blocking_quality_count > 0:
    failed_checks = [
        row["check_id"]
        for row in blocking_quality_df.select("check_id").collect()
    ]

    raise RuntimeError(
        "Gold publishing was blocked by failed Silver quality checks: "
        f"{failed_checks}"
    )

print(
    "Silver quality gate confirmed for execution timestamp: "
    f"{latest_quality_timestamp}"
)


## 5. Read validated Silver datasets

In [ ]:
silver_dataframes = {}

for dataset_name, table_name in SILVER_TABLES.items():
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Silver source table not found for '{dataset_name}': "
            f"{table_name}"
        )

    dataframe = spark.table(table_name)

    if len(dataframe.columns) != len(set(dataframe.columns)):
        raise ValueError(
            f"Silver dataset '{dataset_name}' contains duplicate columns."
        )

    silver_dataframes[dataset_name] = dataframe
    print(f"Validated Silver dataset loaded: {dataset_name}")

header_silver_df = silver_dataframes["inspection_header"]
responses_silver_df = silver_dataframes["inspection_responses"]
requirements_silver_df = silver_dataframes["inspection_requirements"]


## 6. Prepare Gold inspection headers

The Gold header table keeps one row per inspection and exposes the business context required by the reporting model.


In [ ]:
require_columns(
    dataframe=header_silver_df,
    required_columns={
        "inspection_id",
        "inspection_date",
        "inspection_year",
        "site",
        "shift",
        "segment",
        "business_unit_area",
        "line",
        "workstation",
        "activity_type",
        "contract_type",
        "job_role",
        "unsafe_condition",
        "safety_tag_opened",
        "near_miss",
        "audit_type",
        "observed_people_count",
        "header_key",
    },
    dataset_name="inspection_header_silver",
)

header_gold_df = (
    header_silver_df
    .drop("source_system", "source_list")
    .dropDuplicates(["inspection_id"])
    .withColumn("published_timestamp", F.current_timestamp())
)

print("Gold inspection-header dataset prepared successfully.")


## 7. Prepare Gold requirements

The Gold requirements table retains active and historical questionnaire metadata. The `active` field allows report consumers to distinguish current from inactive requirements.


In [ ]:
require_columns(
    dataframe=requirements_silver_df,
    required_columns={
        "requirement_key",
        "site",
        "methodology",
        "questionnaire_version",
        "theme",
        "category_id",
        "category_name_pt_br",
        "category_name_es_ar",
        "question_id",
        "question_number",
        "requirement_text",
        "active",
    },
    dataset_name="inspection_requirements_silver",
)

requirements_gold_df = (
    requirements_silver_df
    .drop("source_system", "source_list")
    .dropDuplicates(["requirement_key"])
    .withColumn("published_timestamp", F.current_timestamp())
)

print("Gold requirements dataset prepared successfully.")


## 8. Prepare Gold inspection responses

The Gold response table keeps one row per inspection-question combination and includes the questionnaire metadata added in the Silver layer.


In [ ]:
require_columns(
    dataframe=responses_silver_df,
    required_columns={
        "response_key",
        "inspection_id",
        "inspection_date",
        "site",
        "audit_type",
        "questionnaire_version",
        "question_id",
        "question_number",
        "theme",
        "category_id",
        "safe_act",
        "unsafe_act",
    },
    dataset_name="inspection_responses_silver",
)

responses_gold_df = (
    responses_silver_df
    .drop("source_system", "source_list")
    .dropDuplicates(["response_key"])
    .withColumn("total_act", F.col("safe_act") + F.col("unsafe_act"))
    .withColumn(
        "has_unsafe_act",
        F.col("unsafe_act") > F.lit(0),
    )
    .withColumn("published_timestamp", F.current_timestamp())
)

print("Gold inspection-response dataset prepared successfully.")


## 9. Create the classification fact table

Each response is expanded into two rows:

- `safe`, containing the safe-observation quantity;
- `unsafe`, containing the unsafe-observation quantity.

This fan-out structure supports straightforward Power BI measures, legends, and comparisons by classification.


In [ ]:
classification_base_df = responses_gold_df.select(
    "response_key",
    "inspection_id",
    "inspection_date",
    "site",
    "audit_type",
    "questionnaire_version",
    "question_id",
    "question_number",
    "theme",
    "category_id",
    F.coalesce(F.col("safe_act"), F.lit(0)).cast("int").alias("safe_act"),
    F.coalesce(F.col("unsafe_act"), F.lit(0)).cast("int").alias("unsafe_act"),
)

safe_classification_df = classification_base_df.select(
    "response_key",
    "inspection_id",
    "inspection_date",
    "site",
    "audit_type",
    "questionnaire_version",
    "question_id",
    "question_number",
    "theme",
    "category_id",
    F.lit("safe").alias("classification"),
    F.col("safe_act").alias("act_count"),
)

unsafe_classification_df = classification_base_df.select(
    "response_key",
    "inspection_id",
    "inspection_date",
    "site",
    "audit_type",
    "questionnaire_version",
    "question_id",
    "question_number",
    "theme",
    "category_id",
    F.lit("unsafe").alias("classification"),
    F.col("unsafe_act").alias("act_count"),
)

classification_gold_df = (
    safe_classification_df
    .unionByName(unsafe_classification_df)
    .withColumn(
        "classification_key",
        F.concat_ws(
            "_",
            F.col("response_key"),
            F.col("classification"),
        ),
    )
    .withColumn("published_timestamp", F.current_timestamp())
)

print("Gold classification dataset created successfully.")


## 10. Validate Gold schemas and analytical expectations

In [ ]:
gold_dataframes = {
    "inspection_header": {
        "df": header_gold_df,
        "target_table": GOLD_TABLES["inspection_header"],
        "key_column": "inspection_id",
    },
    "inspection_responses": {
        "df": responses_gold_df,
        "target_table": GOLD_TABLES["inspection_responses"],
        "key_column": "response_key",
    },
    "inspection_requirements": {
        "df": requirements_gold_df,
        "target_table": GOLD_TABLES["inspection_requirements"],
        "key_column": "requirement_key",
    },
    "inspection_classification": {
        "df": classification_gold_df,
        "target_table": GOLD_TABLES["inspection_classification"],
        "key_column": "classification_key",
    },
}

REQUIRED_GOLD_COLUMNS = {
    "inspection_header": {
        "inspection_id",
        "inspection_date",
        "site",
        "audit_type",
        "published_timestamp",
    },
    "inspection_responses": {
        "response_key",
        "inspection_id",
        "question_id",
        "theme",
        "safe_act",
        "unsafe_act",
        "total_act",
        "published_timestamp",
    },
    "inspection_requirements": {
        "requirement_key",
        "question_id",
        "theme",
        "active",
        "published_timestamp",
    },
    "inspection_classification": {
        "classification_key",
        "response_key",
        "inspection_id",
        "classification",
        "act_count",
        "theme",
        "published_timestamp",
    },
}

publishing_summary = []
record_counts = {}

for dataset_name, dataset_info in gold_dataframes.items():
    dataframe = dataset_info["df"]
    key_column = dataset_info["key_column"]

    require_columns(
        dataframe=dataframe,
        required_columns=REQUIRED_GOLD_COLUMNS[dataset_name],
        dataset_name=f"{dataset_name}_gold",
    )

    duplicate_key_count = (
        dataframe.groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if duplicate_key_count > 0:
        raise ValueError(
            f"Gold dataset '{dataset_name}' contains duplicate "
            f"values in {key_column}."
        )

    null_key_count = dataframe.filter(
        F.col(key_column).isNull()
        | (F.trim(F.col(key_column).cast("string")) == "")
    ).count()

    if null_key_count > 0:
        raise ValueError(
            f"Gold dataset '{dataset_name}' contains null or empty "
            f"values in {key_column}."
        )

    record_count = dataframe.count()
    record_counts[dataset_name] = record_count

    publishing_summary.append(
        {
            "dataset": dataset_name,
            "target_table": dataset_info["target_table"],
            "records": record_count,
            "columns": len(dataframe.columns),
            "status": "ready",
        }
    )

expected_classification_count = record_counts["inspection_responses"] * 2

if record_counts["inspection_classification"] != expected_classification_count:
    raise ValueError(
        "Classification table must contain exactly two rows per response. "
        f"Expected {expected_classification_count}; generated "
        f"{record_counts['inspection_classification']}."
    )

invalid_classification_count = classification_gold_df.filter(
    ~F.col("classification").isin("safe", "unsafe")
    | F.col("act_count").isNull()
    | (F.col("act_count") < 0)
).count()

if invalid_classification_count > 0:
    raise ValueError(
        "Classification dataset contains invalid classification or act_count values."
    )

publishing_summary_df = spark.createDataFrame(publishing_summary)
display(publishing_summary_df.orderBy("dataset"))

print("Gold schemas and analytical expectations validated successfully.")


## 11. Persist Gold Delta tables

In [ ]:
load_summary = []

for dataset_name, dataset_info in gold_dataframes.items():
    dataframe = dataset_info["df"]
    target_table = dataset_info["target_table"]
    record_count = record_counts[dataset_name]

    print(f"Publishing Gold dataset '{dataset_name}' to {target_table}...")

    (
        dataframe.write.format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", OVERWRITE_SCHEMA)
        .saveAsTable(target_table)
    )

    load_summary.append(
        {
            "dataset": dataset_name,
            "target_table": target_table,
            "records": record_count,
            "status": "published",
        }
    )

    print(f"Gold table published successfully: {record_count} records.")

load_summary_df = spark.createDataFrame(load_summary)
display(load_summary_df.orderBy("dataset"))


## 12. Post-write verification

The persisted row counts are compared with the prepared in-memory datasets to confirm that the complete result was published.


In [ ]:
verification_results = []

for dataset_name, target_table in GOLD_TABLES.items():
    if not spark.catalog.tableExists(target_table):
        raise RuntimeError(
            f"Gold table was not created for '{dataset_name}': {target_table}"
        )

    persisted_count = spark.table(target_table).count()
    expected_count = record_counts[dataset_name]
    status = "matched" if persisted_count == expected_count else "mismatch"

    verification_results.append(
        {
            "dataset": dataset_name,
            "target_table": target_table,
            "expected_records": expected_count,
            "persisted_records": persisted_count,
            "status": status,
        }
    )

    if persisted_count != expected_count:
        raise RuntimeError(
            f"Gold row-count verification failed for '{dataset_name}'. "
            f"Expected {expected_count}; persisted {persisted_count}."
        )

verification_df = spark.createDataFrame(verification_results)
display(verification_df.orderBy("dataset"))

print("E-SMAT analytical publishing completed successfully.")


## Published analytical model

```text
inspection_header
        │
        │ inspection_id
        ▼
inspection_responses
        │
        ├── question_id ──► inspection_requirements
        │
        └── response_key ─► inspection_classification
```

The Gold layer is now ready for Power BI modeling and dashboard development.
